# MySQL Database

`MySQLConnection` reads CIM models from a MySQL database built with the ANSI SQL schema generated by [UCAiug CIMTool](https://cimtool.ucaiug.io). The connection is read-only.

The schema conventions it relies on:

* Each CIM class is its own table. A subclass row shares the `"mRID"` of its parent row (a Terminal has rows in `IdentifiedObject`, `ACDCTerminal` and `Terminal`). The class hierarchy is read from the `mRID -> mRID` foreign keys in `information_schema`.
* Associations are columns named after the role (e.g. `"Terminal"."ConductingEquipment"`) holding the target's mRID. Only the single-valued end is stored, so list ends such as `ConductingEquipment.Terminals` are rebuilt by reverse lookup using the data profile.
* Enumerations are stored as literal values (e.g. `'ABC'`) and converted to the data profile enum.
* Booleans are stored as 0/1 and numbers in SI units.
* Identifiers are double-quoted, so each session enables `ANSI_QUOTES`.

mRIDs are matched exactly as stored in the database (e.g. `_49AD8E07-...` with the leading underscore).

## Connection Parameters

The connection is configured with environment variables:

In [ ]:
import os
os.environ['CIMG_CIM_PROFILE'] = 'rc4_2021'
os.environ['CIMG_HOST'] = '127.0.0.1'
os.environ['CIMG_PORT'] = '3306'
os.environ['CIMG_USERNAME'] = 'root'
os.environ['CIMG_PASSWORD'] = 'password'
os.environ['CIMG_DATABASE'] = 'tier5_ieee13'
os.environ['CIMG_USE_UNITS'] = 'true'

In [ ]:
from cimgraph.databases import MySQLConnection
from cimgraph.models import FeederModel
from cimgraph import utils

database = MySQLConnection()
cim = database.cim

## Building a Feeder Model

`get_object` finds the concrete class of an mRID by searching the class tables. `FeederModel` then loads all equipment in the feeder with their terminals and connectivity nodes.

In [ ]:
feeder = database.get_object('_49AD8E07-3BF9-A4E2-CB8F-C3722F837B62')
network = FeederModel(connection=database, container=feeder, distributed=False)

In [ ]:
network.get_all_edges(cim.ACLineSegment)
network.pprint(cim.ACLineSegment)

In [ ]:
# Expand the graph to all objects reachable from the feeder
utils.get_all_data(network)